# Aufteilung der Medienbeiträge

Dieses Notebook teilt die Datei `FSI_Heilbronn_Medienbeiträge (1).json` in drei Schritten auf. Die Datei liegt im übergeordneten Ordner, unabhängig von dessen Namen.

1. **Sprache:** Deutsch und Französisch
2. **Kanal:** Zeitungen und klassische Medien gegenüber Social Media
3. **Social-Media-Quelle:** YouTube, Instagram, Facebook und die übrigen Plattformen

Beiträge, die weder eine Zeitung noch eine Social-Media-Plattform sind, bleiben in der Gruppe **Andere Webquellen**. Dazu gehören Foren, Blogs, Kommentare unter Artikeln und Seiten, die der Datensatz nur als `Other` führt. So bleibt jede Zeile einer Gruppe zugeordnet.

### Zuordnungsregeln

- **Deutsch:** `language_code` beginnt mit `de` (`de`, `de-DE`, `de-AT`, `de-CH`).
- **Französisch:** `language_code` beginnt mit `fr`.
- **Social Media:** die Quelle ist eine bekannte Plattform. Mastodon-Instanzen stehen im Datensatz als `Microblog` und werden zu Mastodon gezählt. Bluesky bleibt eigene Quelle.
- **Zeitungen und klassische Medien:** `media_type_name` ist `News` und die Quelle ist keine Social-Media-Plattform.
- **Andere Webquellen:** alles Übrige.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FuncFormatter

DATEN_NAME = "FSI_Heilbronn_Medienbeiträge (1).json"


def datenordner():
    """Ordner über diesem Notebook, unabhängig vom Ordnernamen."""
    hier = Path.cwd().resolve()
    for ordner in (hier.parent, hier, hier.parent.parent):
        if (ordner / DATEN_NAME).is_file():
            return ordner
    return hier.parent


DATEN = datenordner() / DATEN_NAME

PLATTFORM_DOMAINS = (
    ("YouTube", ("youtube.com", "youtu.be")),
    ("Instagram", ("instagram.com",)),
    ("Facebook", ("facebook.com",)),
    ("Reddit", ("reddit.com",)),
    ("LinkedIn", ("linkedin.com",)),
    ("TikTok", ("tiktok.com",)),
    ("Telegram", ("t.me", "telegram.me", "telegram.org")),
    ("Flickr", ("flickr.com",)),
    ("Pinterest", ("pinterest.com",)),
    ("Tumblr", ("tumblr.com",)),
    ("Bluesky", ("bsky.app",)),
    ("Vimeo", ("vimeo.com",)),
    ("X", ("twitter.com", "x.com")),
)

KANAL_REIHENFOLGE = (
    "Zeitungen und klassische Medien",
    "Social Media",
    "Andere Webquellen",
)
SPRACHEN = ("Deutsch", "Französisch")
FARBEN = {"Deutsch": "#e07a3d", "Französisch": "#3d7ec9"}


def host_aus_quelle(quelle):
    host = str(quelle or "").strip().lower()
    if host.startswith("www."):
        host = host[4:]
    return host


def host_ist(host, domain):
    return host == domain or host.endswith("." + domain)


def plattform_bestimmen(quelle, medientyp):
    host = host_aus_quelle(quelle)
    for name, domains in PLATTFORM_DOMAINS:
        if any(host_ist(host, domain) for domain in domains):
            return name
    if medientyp == "Microblog":
        return "Mastodon"
    if medientyp in {"YouTube Video", "YouTube Comments"}:
        return "YouTube"
    return None


def sprache_bestimmen(code):
    code = str(code or "").lower()
    if code.startswith("de"):
        return "Deutsch"
    if code.startswith("fr"):
        return "Französisch"
    return "Andere"


def kanal_bestimmen(medientyp, plattform):
    if isinstance(plattform, str) and plattform:
        return "Social Media"
    if medientyp == "News":
        return "Zeitungen und klassische Medien"
    return "Andere Webquellen"


def mit_summe(tabelle):
    ergebnis = tabelle.copy()
    ergebnis["Summe"] = ergebnis.sum(axis=1)
    ergebnis.loc["Summe"] = ergebnis.sum(axis=0)
    return ergebnis.astype(int)


def als_text(tabelle):
    def zelle(wert):
        return f"{int(wert):,}".replace(",", ".")

    if isinstance(tabelle, pd.Series):
        return tabelle.map(zelle)
    return tabelle.apply(lambda spalte: spalte.map(zelle))


def achsen_tausender(achse, _position):
    return f"{int(achse):,}".replace(",", ".")


df = pd.read_json(DATEN, orient="records")
sprache = [sprache_bestimmen(code) for code in df["language_code"]]
plattformen = [
    plattform_bestimmen(quelle, medientyp)
    for quelle, medientyp in zip(df["source_name"], df["media_type_name"])
]
kanaele = [
    kanal_bestimmen(medientyp, plattform)
    for medientyp, plattform in zip(df["media_type_name"], plattformen)
]
df["sprache"] = pd.Series(sprache, index=df.index, dtype="object")
df["plattform"] = pd.Series(plattformen, index=df.index, dtype="object")
df["kanal"] = pd.Series(kanaele, index=df.index, dtype="object")

vorn = ["sprache", "kanal", "plattform", "media_type_name", "source_name", "language_code", "published_at", "title"]
df = df[vorn + [spalte for spalte in df.columns if spalte not in vorn]]

df[vorn].head(8)


## Sprache

Deutsch und Französisch sind die beiden Sprachen der Challenge. Die übrigen Sprachcodes bleiben als **Andere** sichtbar, damit die Summe der Datei aufgeht. Die weiteren Tabellen zeigen nur Deutsch und Französisch.


In [ ]:
sprache_anzahl = df["sprache"].value_counts().rename("Beiträge")
sprache_anzahl = sprache_anzahl.reindex(["Deutsch", "Französisch", "Andere"])
anzeige_sprache = als_text(sprache_anzahl.to_frame())
anzeige_sprache


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
balken = sprache_anzahl.loc[list(SPRACHEN)]
farben = [FARBEN[name] for name in balken.index]
balken.plot(kind="bar", ax=ax, color=farben, width=0.7)
ax.set_title("Beiträge nach Sprache")
ax.set_xlabel("")
ax.set_ylabel("Beiträge")
ax.yaxis.set_major_formatter(FuncFormatter(achsen_tausender))
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
plt.show()


## Zeitungen und Social Media

Innerhalb von Deutsch und Französisch teilt `kanal` die Beiträge in Zeitungen, Social Media und andere Webquellen.


In [ ]:

teil = df[df["sprache"].isin(SPRACHEN)]
kanal_anzahl = (
    pd.crosstab(teil["kanal"], teil["sprache"])
    .reindex(index=list(KANAL_REIHENFOLGE), columns=list(SPRACHEN), fill_value=0)
)
kanal_mit_summe = mit_summe(kanal_anzahl)
als_text(kanal_mit_summe)


In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 4.4))
kanal_anzahl.plot(kind="bar", ax=ax, color=[FARBEN[name] for name in kanal_anzahl.columns], width=0.75)
ax.set_title("Kanal nach Sprache")
ax.set_xlabel("")
ax.set_ylabel("Beiträge")
ax.yaxis.set_major_formatter(FuncFormatter(achsen_tausender))
ax.tick_params(axis="x", rotation=12)
ax.legend(title="")
fig.tight_layout()
plt.show()


In [ ]:

def ausschnitt(sprache, kanal=None, plattform=None):
    """Liefert die Zeilen einer Sprache, optional weiter nach Kanal oder Plattform."""
    auswahl = df[df["sprache"].eq(sprache)]
    if kanal is not None:
        auswahl = auswahl[auswahl["kanal"].eq(kanal)]
    if plattform is not None:
        auswahl = auswahl[auswahl["plattform"].eq(plattform)]
    return auswahl


rest = df[df["sprache"].isin(SPRACHEN) & df["kanal"].eq("Andere Webquellen")]
andere_webquellen = (
    pd.crosstab(rest["media_type_name"], rest["sprache"])
    .reindex(columns=list(SPRACHEN), fill_value=0)
    .sort_values("Deutsch", ascending=False)
)
als_text(mit_summe(andere_webquellen))


## Social-Media-Quellen

Nur die Zeilen mit Kanal **Social Media**. Dieselbe Quelle fasst alle Beitragsarten der Plattform zusammen. Bei YouTube liegen Videos und Kommentare darunter.


In [ ]:

sozial = df[df["kanal"].eq("Social Media") & df["sprache"].isin(SPRACHEN)]
plattform_anzahl = pd.crosstab(sozial["plattform"], sozial["sprache"]).reindex(columns=list(SPRACHEN), fill_value=0)
plattform_anzahl["Summe"] = plattform_anzahl.sum(axis=1)
plattform_anzahl = plattform_anzahl.sort_values("Summe", ascending=False)
plattform_mit_summe = plattform_anzahl.copy()
plattform_mit_summe.loc["Summe"] = plattform_mit_summe.sum(axis=0)
als_text(plattform_mit_summe.astype(int))


In [ ]:
plot_plattform = plattform_anzahl.drop(columns="Summe").sort_values("Deutsch", ascending=True)
fig, ax = plt.subplots(figsize=(8.4, 5.6))
plot_plattform.plot(
    kind="barh",
    ax=ax,
    color=[FARBEN[name] for name in plot_plattform.columns],
    width=0.75,
)
ax.set_title("Social-Media-Quellen nach Sprache")
ax.set_xlabel("Beiträge")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(FuncFormatter(achsen_tausender))
ax.legend(title="")
fig.tight_layout()
plt.show()


In [ ]:

youtube = df[df["plattform"].eq("YouTube") & df["sprache"].isin(SPRACHEN)].copy()
youtube["beitragsart"] = youtube["media_type_name"].map(
    {"YouTube Video": "Video", "YouTube Comments": "Kommentar"}
).fillna("Sonstige")
youtube_anzahl = pd.crosstab(youtube["beitragsart"], youtube["sprache"]).reindex(
    index=["Video", "Kommentar", "Sonstige"],
    columns=list(SPRACHEN),
    fill_value=0,
)
youtube_anzahl = youtube_anzahl.loc[youtube_anzahl.sum(axis=1).gt(0)]
als_text(mit_summe(youtube_anzahl))


In [ ]:
beispiele = (
    df[df["kanal"].eq("Social Media") & df["sprache"].isin(SPRACHEN)]
    .sort_values(["plattform", "sprache"])
    .groupby(["plattform", "sprache"], as_index=False)
    .head(1)
)
beispiele[["plattform", "sprache", "media_type_name", "source_name", "title"]]
